In [0]:
%run ./00_config

In [0]:
# ============================================================
# GOLD DAILY SALES
# ============================================================
# Source:
#   billing.sales_data.billing_silver
#
# Target:
#   billing.sales_data.gold_daily_sales
#
# Grain:
#   orderDate + storeName + source_date
#
# IMPORTANT:
#   This version preserves the EXISTING Gold table schema.
#
# Business logic:
#   - NO orderStatus filter
#   - Exclude Ho Marlboro
#   - Exclude Dummy Store --- For Testing Only
#   - total_orders      = COUNT(DISTINCT invoice)
#   - unique_customers  = COUNT(DISTINCT customerNumber)
#   - total_net_sales   = SUM(totalProductPrice)
#   - total_tax_collected = SUM(GSTAmount)
#   - total_discounts   = SUM(discountAmount)
#   - total_delivery_fees = SUM(deliveryFee)
#   - avg_order_value   = total_net_sales / total_orders
#   - payment metrics based on paymentMethod
#
# Write strategy:
#   - Atomic Delta replaceWhere
#   - No DELETE + APPEND
#   - Idempotent for a target date
#   - Existing source_date STRING is preserved
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    DateType,
    LongType,
    DoubleType
)

from datetime import datetime, timezone
import uuid


# ============================================================
# CONFIGURATION
# ============================================================

SOURCE_TABLE = "billing.sales_data.billing_silver"
TARGET_TABLE = "billing.sales_data.gold_daily_sales"

TARGET_DATE = "2026-10-03"

PIPELINE_RUN_ID = str(uuid.uuid4())

EXCLUDED_STORES = [
    "ho marlboro",
    "dummy store --- for testing only"
]


# ============================================================
# HELPERS
# ============================================================

def log(message):
    print(f"[GOLD_DAILY_SALES] {message}")


def fail(message):
    raise RuntimeError(
        f"[GOLD_DAILY_SALES] {message}"
    )


def table_exists(table_name):
    return spark.catalog.tableExists(table_name)


# ============================================================
# START
# ============================================================

START_TIME = datetime.now(timezone.utc)

log("=" * 80)
log("PRODUCTION GOLD DAILY SALES")
log("=" * 80)

log(f"Pipeline run ID : {PIPELINE_RUN_ID}")
log(f"Target date     : {TARGET_DATE}")
log(f"Source table    : {SOURCE_TABLE}")
log(f"Target table    : {TARGET_TABLE}")


# ============================================================
# VALIDATE SOURCE TABLE
# ============================================================

if not table_exists(SOURCE_TABLE):
    fail(
        f"Source table does not exist: {SOURCE_TABLE}"
    )


silver_df = spark.table(SOURCE_TABLE)


# ============================================================
# REQUIRED SOURCE COLUMNS
# ============================================================

required_columns = [
    "source_date",
    "orderDate",
    "storeName",
    "invoice",
    "customerNumber",
    "totalProductPrice",
    "GSTAmount",
    "discountAmount",
    "deliveryFee",
    "paymentMethod"
]

missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in silver_df.columns
]

if missing_columns:
    fail(
        f"Missing required Silver columns: "
        f"{missing_columns}"
    )


# ============================================================
# READ TARGET-DATE SILVER DATA
# ============================================================

silver_target = (
    silver_df
    .filter(
        F.col("source_date").cast("string") == TARGET_DATE
    )
)

silver_input_count = silver_target.count()

log(
    f"Silver rows for {TARGET_DATE}: "
    f"{silver_input_count}"
)

if silver_input_count == 0:
    fail(
        f"No Silver data found for "
        f"source_date={TARGET_DATE}"
    )


# ============================================================
# APPLY BUSINESS STORE EXCLUSIONS
#
# Case-insensitive + whitespace-safe
# ============================================================

silver_filtered = (
    silver_target
    .filter(
        ~F.lower(
            F.trim(
                F.col("storeName")
            )
        ).isin(EXCLUDED_STORES)
    )
)

filtered_count = silver_filtered.count()

log(
    f"Rows after store exclusions: "
    f"{filtered_count}"
)

if filtered_count == 0:
    fail(
        "All Silver rows were removed by "
        "store exclusion rules"
    )


# ============================================================
# PREPARE SOURCE COLUMNS
# ============================================================

prepared = (
    silver_filtered

    # Business transaction date
    .withColumn(
        "_orderDate",
        F.to_date(F.col("orderDate"))
    )

    # Sales
    .withColumn(
        "_sales",
        F.col("totalProductPrice").cast("double")
    )

    # Tax
    .withColumn(
        "_tax",
        F.col("GSTAmount").cast("double")
    )

    # Discounts
    .withColumn(
        "_discount",
        F.col("discountAmount").cast("double")
    )

    # Delivery
    .withColumn(
        "_delivery_fee",
        F.col("deliveryFee").cast("double")
    )

    # Invoice
    .withColumn(
        "_invoice",
        F.trim(
            F.col("invoice").cast("string")
        )
    )

    # Customer
    .withColumn(
        "_customer",
        F.trim(
            F.col("customerNumber").cast("string")
        )
    )

    # Normalized payment method
    .withColumn(
        "_payment_method",
        F.upper(
            F.trim(
                F.col("paymentMethod").cast("string")
            )
        )
    )
)


# ============================================================
# AGGREGATE
# ============================================================

gold_df = (
    prepared
    .groupBy(
        "_orderDate",
        "storeName"
    )
    .agg(

        # ----------------------------------------------------
        # ORDERS
        # ----------------------------------------------------

        F.countDistinct(
            F.when(
                F.col("_invoice").isNotNull() &
                (F.col("_invoice") != ""),
                F.col("_invoice")
            )
        ).cast("long").alias(
            "total_orders"
        ),

        # ----------------------------------------------------
        # UNIQUE CUSTOMERS
        # ----------------------------------------------------

        F.countDistinct(
            F.when(
                F.col("_customer").isNotNull() &
                (F.col("_customer") != ""),
                F.col("_customer")
            )
        ).cast("long").alias(
            "unique_customers"
        ),

        # ----------------------------------------------------
        # SALES
        # ----------------------------------------------------

        F.sum(
            F.coalesce(
                F.col("_sales"),
                F.lit(0.0)
            )
        ).cast("double").alias(
            "total_net_sales"
        ),

        # ----------------------------------------------------
        # TAX
        # ----------------------------------------------------

        F.sum(
            F.coalesce(
                F.col("_tax"),
                F.lit(0.0)
            )
        ).cast("double").alias(
            "total_tax_collected"
        ),

        # ----------------------------------------------------
        # DISCOUNTS
        # ----------------------------------------------------

        F.sum(
            F.coalesce(
                F.col("_discount"),
                F.lit(0.0)
            )
        ).cast("double").alias(
            "total_discounts"
        ),

        # ----------------------------------------------------
        # DELIVERY FEES
        # ----------------------------------------------------

        F.sum(
            F.coalesce(
                F.col("_delivery_fee"),
                F.lit(0.0)
            )
        ).cast("double").alias(
            "total_delivery_fees"
        ),

        # ----------------------------------------------------
        # CASH
        # ----------------------------------------------------

        F.sum(
            F.when(
                F.col("_payment_method") == "CASH",
                F.coalesce(
                    F.col("_sales"),
                    F.lit(0.0)
                )
            ).otherwise(
                F.lit(0.0)
            )
        ).cast("double").alias(
            "total_cash"
        ),

        # ----------------------------------------------------
        # CARD
        # ----------------------------------------------------

        F.sum(
            F.when(
                F.col("_payment_method") == "CARD",
                F.coalesce(
                    F.col("_sales"),
                    F.lit(0.0)
                )
            ).otherwise(
                F.lit(0.0)
            )
        ).cast("double").alias(
            "total_card"
        ),

        # ----------------------------------------------------
        # UPI
        # ----------------------------------------------------

        F.sum(
            F.when(
                F.col("_payment_method") == "UPI",
                F.coalesce(
                    F.col("_sales"),
                    F.lit(0.0)
                )
            ).otherwise(
                F.lit(0.0)
            )
        ).cast("double").alias(
            "total_upi"
        ),

        # ----------------------------------------------------
        # CREDIT
        # ----------------------------------------------------

        F.sum(
            F.when(
                F.col("_payment_method") == "CREDIT",
                F.coalesce(
                    F.col("_sales"),
                    F.lit(0.0)
                )
            ).otherwise(
                F.lit(0.0)
            )
        ).cast("double").alias(
            "total_credit"
        )
    )

    .withColumnRenamed(
        "_orderDate",
        "orderDate"
    )
)


# ============================================================
# CALCULATE AOV
# ============================================================

gold_df = (
    gold_df
    .withColumn(
        "avg_order_value",
        F.when(
            F.col("total_orders") > 0,
            F.col("total_net_sales") /
            F.col("total_orders")
        )
        .otherwise(
            F.lit(0.0)
        )
        .cast("double")
    )
)


# ============================================================
# ADD SOURCE DATE
#
# IMPORTANT:
# source_date = PIPELINE SNAPSHOT DATE
#
# It must NOT be derived from orderDate.
# ============================================================

gold_df = (
    gold_df
    .withColumn(
        "source_date",
        F.lit(TARGET_DATE).cast("string")
    )
)


# ============================================================
# SELECT EXACT TARGET SCHEMA
#
# DO NOT ADD:
#   pipeline_run_id
#   processed_at
#
# because the existing Gold table does not contain them.
# ============================================================

gold_df = (
    gold_df
    .select(
        F.col("orderDate").cast("date").alias(
            "orderDate"
        ),

        F.col("storeName").cast("string").alias(
            "storeName"
        ),

        F.col("total_orders").cast("long").alias(
            "total_orders"
        ),

        F.col("unique_customers").cast("long").alias(
            "unique_customers"
        ),

        F.col("total_net_sales").cast("double").alias(
            "total_net_sales"
        ),

        F.col("total_tax_collected").cast("double").alias(
            "total_tax_collected"
        ),

        F.col("total_discounts").cast("double").alias(
            "total_discounts"
        ),

        F.col("total_delivery_fees").cast("double").alias(
            "total_delivery_fees"
        ),

        F.col("avg_order_value").cast("double").alias(
            "avg_order_value"
        ),

        F.col("total_cash").cast("double").alias(
            "total_cash"
        ),

        F.col("total_card").cast("double").alias(
            "total_card"
        ),

        F.col("total_upi").cast("double").alias(
            "total_upi"
        ),

        F.col("total_credit").cast("double").alias(
            "total_credit"
        ),

        F.col("source_date").cast("string").alias(
            "source_date"
        )
    )
)


# ============================================================
# VALIDATE GOLD OUTPUT
# ============================================================

gold_count = gold_df.count()

log(
    f"Gold rows generated: {gold_count}"
)

if gold_count == 0:
    fail(
        "Gold aggregation produced zero rows"
    )


# ============================================================
# VALIDATE GOLD SCHEMA
# ============================================================

expected_columns = [
    "orderDate",
    "storeName",
    "total_orders",
    "unique_customers",
    "total_net_sales",
    "total_tax_collected",
    "total_discounts",
    "total_delivery_fees",
    "avg_order_value",
    "total_cash",
    "total_card",
    "total_upi",
    "total_credit",
    "source_date"
]

actual_columns = gold_df.columns

if actual_columns != expected_columns:
    fail(
        "Generated Gold schema does not match expected schema.\n"
        f"Expected: {expected_columns}\n"
        f"Actual  : {actual_columns}"
    )

log("Generated Gold schema validation passed")


# ============================================================
# DUPLICATE GRAIN VALIDATION
# ============================================================

duplicate_groups = (
    gold_df
    .groupBy(
        "orderDate",
        "storeName",
        "source_date"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
    .count()
)

if duplicate_groups > 0:
    fail(
        f"Duplicate Gold grain detected: "
        f"{duplicate_groups} groups"
    )

log(
    "Duplicate grain validation passed"
)


# ============================================================
# EXCLUDED STORE VALIDATION
# ============================================================

excluded_output_rows = (
    gold_df
    .filter(
        F.lower(
            F.trim(
                F.col("storeName")
            )
        ).isin(EXCLUDED_STORES)
    )
    .count()
)

if excluded_output_rows > 0:
    fail(
        "Excluded stores found in Gold output: "
        f"{excluded_output_rows}"
    )

log(
    "Excluded-store validation passed"
)


# ============================================================
# SILVER -> GOLD SALES RECONCILIATION
# ============================================================

silver_sales_row = (
    silver_filtered
    .agg(
        F.sum(
            F.col("totalProductPrice")
            .cast("double")
        ).alias("sales")
    )
    .first()
)

silver_sales = float(
    silver_sales_row["sales"] or 0.0
)


gold_sales_row = (
    gold_df
    .agg(
        F.sum(
            F.col("total_net_sales")
        ).alias("sales")
    )
    .first()
)

gold_sales = float(
    gold_sales_row["sales"] or 0.0
)


sales_difference = abs(
    silver_sales - gold_sales
)

log(
    f"Silver filtered sales : "
    f"{silver_sales:.4f}"
)

log(
    f"Gold aggregated sales : "
    f"{gold_sales:.4f}"
)

log(
    f"Sales difference      : "
    f"{sales_difference:.4f}"
)


if sales_difference > 0.01:
    fail(
        "Silver -> Gold sales reconciliation failed. "
        f"Difference={sales_difference:.4f}"
    )

log(
    "Silver -> Gold sales reconciliation passed"
)


# ============================================================
# TARGET TABLE VALIDATION
# ============================================================

if not table_exists(TARGET_TABLE):

    # --------------------------------------------------------
    # FIRST CREATION
    # --------------------------------------------------------

    log(
        "Target table does not exist."
    )

    log(
        "Creating Gold table with exact schema."
    )

    (
        gold_df
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("source_date")
        .saveAsTable(TARGET_TABLE)
    )

    log(
        "Gold table created successfully."
    )


else:

    # --------------------------------------------------------
    # EXISTING TARGET
    # --------------------------------------------------------

    target_df = spark.table(TARGET_TABLE)

    target_schema = target_df.schema

    target_columns = target_df.columns

    log(
        f"Existing target columns: "
        f"{target_columns}"
    )


    # --------------------------------------------------------
    # VERIFY TARGET SCHEMA
    # --------------------------------------------------------

    expected_target_columns = [
        "orderDate",
        "storeName",
        "total_orders",
        "unique_customers",
        "total_net_sales",
        "total_tax_collected",
        "total_discounts",
        "total_delivery_fees",
        "avg_order_value",
        "total_cash",
        "total_card",
        "total_upi",
        "total_credit",
        "source_date"
    ]

    if target_columns != expected_target_columns:

        fail(
            "Existing target table schema does not match "
            "the expected Gold Daily Sales schema.\n"
            f"Expected: {expected_target_columns}\n"
            f"Actual  : {target_columns}"
        )


    # --------------------------------------------------------
    # VERIFY source_date TYPE
    # --------------------------------------------------------

    source_date_field = next(
        (
            field
            for field in target_schema
            if field.name == "source_date"
        ),
        None
    )

    if source_date_field is None:
        fail(
            "Target table does not contain source_date"
        )


    log(
        "Existing source_date type: "
        f"{source_date_field.dataType.simpleString()}"
    )


    # --------------------------------------------------------
    # THIS TABLE CURRENTLY USES STRING
    # --------------------------------------------------------

    if not isinstance(
        source_date_field.dataType,
        StringType
    ):
        fail(
            "Expected existing Gold source_date to be STRING, "
            "but found: "
            f"{source_date_field.dataType.simpleString()}"
        )


    # --------------------------------------------------------
    # ENSURE INCOMING TYPE MATCHES TARGET
    # --------------------------------------------------------

    gold_df = gold_df.withColumn(
        "source_date",
        F.col("source_date").cast("string")
    )


    # --------------------------------------------------------
    # FINAL COLUMN ORDER
    # --------------------------------------------------------

    gold_df = (
        gold_df
        .select(
            "orderDate",
            "storeName",
            "total_orders",
            "unique_customers",
            "total_net_sales",
            "total_tax_collected",
            "total_discounts",
            "total_delivery_fees",
            "avg_order_value",
            "total_cash",
            "total_card",
            "total_upi",
            "total_credit",
            "source_date"
        )
    )


    # --------------------------------------------------------
    # ATOMIC REPLACEMENT
    # --------------------------------------------------------

    replace_predicate = (
        f"source_date = '{TARGET_DATE}'"
    )

    log(
        "Atomically replacing Gold snapshot:"
    )

    log(
        f"replaceWhere = {replace_predicate}"
    )


    (
        gold_df
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            replace_predicate
        )
        .saveAsTable(TARGET_TABLE)
    )


    log(
        "Atomic Gold replacement committed."
    )


# ============================================================
# POST-WRITE VALIDATION
# ============================================================

target_after_write = spark.table(
    TARGET_TABLE
)


post_write_df = (
    target_after_write
    .filter(
        F.col("source_date").cast("string")
        == TARGET_DATE
    )
)


post_write_count = post_write_df.count()


log(
    f"Post-write validation: "
    f"expected={gold_count}, "
    f"actual={post_write_count}"
)


if post_write_count != gold_count:
    fail(
        "Post-write row-count mismatch. "
        f"Expected={gold_count}, "
        f"Actual={post_write_count}"
    )


# ============================================================
# POST-WRITE SALES VALIDATION
# ============================================================

post_write_sales_row = (
    post_write_df
    .agg(
        F.sum(
            F.col("total_net_sales")
        ).alias("sales")
    )
    .first()
)

post_write_sales = float(
    post_write_sales_row["sales"] or 0.0
)


post_write_difference = abs(
    gold_sales - post_write_sales
)


log(
    f"Gold before write sales : "
    f"{gold_sales:.4f}"
)

log(
    f"Gold after write sales  : "
    f"{post_write_sales:.4f}"
)

log(
    f"Post-write difference   : "
    f"{post_write_difference:.4f}"
)


if post_write_difference > 0.01:
    fail(
        "Post-write sales validation failed. "
        f"Difference={post_write_difference:.4f}"
    )


# ============================================================
# FINAL
# ============================================================

END_TIME = datetime.now(timezone.utc)

duration_seconds = (
    END_TIME - START_TIME
).total_seconds()


log("=" * 80)
log("GOLD DAILY SALES COMPLETED SUCCESSFULLY")
log("=" * 80)

log(
    f"Pipeline run ID : {PIPELINE_RUN_ID}"
)

log(
    f"Target date     : {TARGET_DATE}"
)

log(
    f"Gold rows       : {gold_count}"
)

log(
    f"Total sales     : {gold_sales:.2f}"
)

log(
    f"Duration        : {duration_seconds:.2f} seconds"
)

log("=" * 80)

[GOLD_DAILY_SALES] ================================================================================
[GOLD_DAILY_SALES] PRODUCTION GOLD DAILY SALES
[GOLD_DAILY_SALES] ================================================================================
[GOLD_DAILY_SALES] Pipeline run ID : be225b34-8f33-4167-8947-a0424cddf023
[GOLD_DAILY_SALES] Target date     : 2026-10-03
[GOLD_DAILY_SALES] Source table    : billing.sales_data.billing_silver
[GOLD_DAILY_SALES] Target table    : billing.sales_data.gold_daily_sales
[GOLD_DAILY_SALES] Silver rows for 2026-10-03: 52995
[GOLD_DAILY_SALES] Rows after store exclusions: 52995
[GOLD_DAILY_SALES] Gold rows generated: 107
[GOLD_DAILY_SALES] Generated Gold schema validation passed
[GOLD_DAILY_SALES] Duplicate grain validation passed
[GOLD_DAILY_SALES] Excluded-store validation passed
[GOLD_DAILY_SALES] Silver filtered sales : 4496998.8700
[GOLD_DAILY_SALES] Gold aggregated sales : 4496998.8700
[GOLD_DAILY_SALES] Sales difference      : 0.0000
[GOLD_DAIL